# BBPE Tokenizer

## BBPE 训练


In [2]:
from tokenizer import train_bbpe, BBPETokenizer


In [ ]:
corpus_path = "../data/TinyStoriesV2-GPT4-train.txt"
vocab_size = 10000
special_tokens = ["<|endoftext|>"]
special_split_token = "<|endoftext|>"

vocab, merges = train_bbpe(corpus_path, vocab_size, special_tokens, special_split_token.encode("utf-8"))
tokenizer = BBPETokenizer(vocab, merges, special_tokens)
tokenizer.save("../deliverable/TinyStoriesV2-10k")


In [ ]:
corpus_path = "../data/owt_train.txt"
vocab_size = 32000
special_tokens = ["<|endoftext|>"]
special_split_token = "<|endoftext|>"

vocab, merges = train_bbpe(corpus_path, vocab_size, special_tokens, special_split_token.encode("utf-8"))
tokenizer = BBPETokenizer(vocab, merges, special_tokens)
tokenizer.save("../deliverable/owt-32k")


## 验证无损性


In [3]:
test_string_list = [
    "",
    "s",
    "🙃",
    "Hello, how are you?",
    "Héllò hôw are ü? 🙃",
    "Héllò hôw <|endoftext|><|endoftext|> are ü? 🙃<|endoftext|>",
    "Hello, how <|endoftext|><|endoftext|> are you?<|endoftext|>",
    "道可道，非常道。"
]

In [4]:
ts_tokenizer = BBPETokenizer.load("../deliverable/TinyStoriesV2-10k")
for test_string in test_string_list:
    assert ts_tokenizer.decode(ts_tokenizer.encode(test_string)) == test_string, print(test_string)
    

In [5]:
owt_tokenizer = BBPETokenizer.load("../deliverable/owt-32k")
for test_string in test_string_list:
    assert owt_tokenizer.decode(owt_tokenizer.encode(test_string)) == test_string, print(test_string)
    

## Experiments

(a) Sample 10 documents from TinyStories and OpenWebText. Using your previously-trained TinyStories and OpenWebText tokenizers (10K and 32K vocabulary size,respectively), encode these sampled documents into integer IDs. What is each tokenizer’s compression ratio (bytes/token)?

In [1]:
import regex as re
import random

special_split_token = "<|endoftext|>"
special_split_pattern = re.compile(re.escape(special_split_token))


In [6]:

corpus_path = "../data/TinyStoriesV2-GPT4-train.txt"
with open(corpus_path, "r") as f:
    content = f.read()
    ts_documents = re.split(special_split_pattern, content)
print(len(ts_documents))


2717700


In [8]:
byte_cnt = token_cnt = 0
for _ in range(10):
    doc = random.choice(ts_documents)
    byte_cnt += len(list(doc.encode("utf-8")))
    token_cnt += len(ts_tokenizer.encode(doc))
print(byte_cnt / token_cnt)


4.089953271028038


In [9]:
corpus_path = "../data/owt_train.txt"
with open(corpus_path, "r") as f:
    content = f.read()
    owt_documents = re.split(special_split_pattern, content)
print(len(owt_documents))


2399398


In [10]:
byte_cnt = token_cnt = 0
for _ in range(10):
    doc = random.choice(owt_documents)
    byte_cnt += len(list(doc.encode("utf-8")))
    token_cnt += len(owt_tokenizer.encode(doc))
print(byte_cnt / token_cnt)


4.619838056680162


(b) What happens if you tokenize your OpenWebText sample with the TinyStories tokenizer? Compare the compression ratio and/or qualitatively describe what happens.

In [13]:
byte_cnt = token_cnt = 0
for _ in range(10):
    doc = random.choice(ts_documents)
    byte_cnt += len(list(doc.encode("utf-8")))
    token_cnt += len(owt_tokenizer.encode(doc))
print(byte_cnt / token_cnt)


3.939655172413793


In [14]:
byte_cnt = token_cnt = 0
for _ in range(10):
    doc = random.choice(owt_documents)
    byte_cnt += len(list(doc.encode("utf-8")))
    token_cnt += len(ts_tokenizer.encode(doc))
print(byte_cnt / token_cnt)


3.0524220752934825


(c) Estimate the throughput of your tokenizer (e.g., in bytes/second). How long would it take totokenize the Pile dataset (825GB of text)?

In [15]:
import os
import time

def estimate_throughput(tokenizer: BBPETokenizer, data_path: str, sample_size_mb: int = 10) -> None:
    """
    Estimates the tokenizer throughput.

    Args:
        data_path: Path to the text data file.
        vocab_path: Path to the vocabulary file.
        merges_path: Path to the merges file.
        special_tokens_path: Path to the special tokens file.
        sample_size_mb: The size of the sample to use for estimation in megabytes.
    """
    # The from_files classmethod in the provided code doesn't handle special tokens,
    # but the BBPETokenizer.load method does. Let's use that logic.
    # We can simulate the `load` method's behavior.
    file_size = os.path.getsize(data_path)
    sample_size_bytes = sample_size_mb * 1024 * 1024
    if file_size < sample_size_bytes:
        sample_size_bytes = file_size

    print(f"Reading {sample_size_bytes / (1024*1024):.2f} MB sample from {data_path}...")
    with open(data_path, "r", encoding="utf-8", errors="ignore") as f:
        text_sample = f.read(sample_size_bytes)

    print("Starting tokenization...")
    start_time = time.time()
    tokens = tokenizer.encode(text_sample)
    end_time = time.time()

    elapsed_time = end_time - start_time
    # We are interested in the throughput of reading bytes from disk and tokenizing.
    # The number of bytes processed is sample_size_bytes.
    bytes_processed = len(text_sample.encode('utf-8'))
    throughput_bytes_per_sec = bytes_processed / elapsed_time
    
    print(f"Tokenized {len(tokens)} tokens from {bytes_processed / (1024*1024):.2f} MB of text in {elapsed_time:.2f} seconds.")
    print(f"Throughput: {throughput_bytes_per_sec / (1024*1024):.2f} MB/s")

    pile_size_gb = 825
    pile_size_bytes = pile_size_gb * 1024 * 1024 * 1024
    estimated_time_seconds = pile_size_bytes / throughput_bytes_per_sec
    estimated_time_hours = estimated_time_seconds / 3600
    estimated_time_days = estimated_time_hours / 24

    print(f"Estimated time to tokenize the Pile dataset ({pile_size_gb} GB): {estimated_time_hours:.2f} hours ({estimated_time_days:.2f} days)")
    

In [18]:
estimate_throughput(ts_tokenizer, "../data/TinyStoriesV2-GPT4-valid.txt")

Reading 10.00 MB sample from ../data/TinyStoriesV2-GPT4-valid.txt...
Starting tokenization...
Tokenized 2547736 tokens from 10.00 MB of text in 20.32 seconds.
Throughput: 0.49 MB/s
Estimated time to tokenize the Pile dataset (825 GB): 476.68 hours (19.86 days)


In [19]:
estimate_throughput(owt_tokenizer, "../data/owt_valid.txt")

Reading 10.00 MB sample from ../data/owt_valid.txt...
Starting tokenization...
Tokenized 2443398 tokens from 10.10 MB of text in 23.09 seconds.
Throughput: 0.44 MB/s
Estimated time to tokenize the Pile dataset (825 GB): 536.30 hours (22.35 days)


(d) Using your TinyStories and OpenWebText tokenizers, encode the respective training and development datasets into a sequence of integer token IDs. We’ll use this later to train our language model. We recommend serializing the token IDs as a NumPy array of datatype uint16. Why is uint16 an appropriate choice?

A：因为 uint16 可以表示的范围是 0 到 65535，而 BPE 编码后的 token ID 通常在这个范围内。